# Chuẩn hóa văn bản, công thức toán học và khử trùng lặp dữ liệu

> **Môn học:** Nhập môn Khoa học Dữ liệu — HCMUS  
> **Milestone 2:** Bóc tách Cấu trúc Tài liệu & Chuẩn Hóa Phân Cấp  
> **Thành viên phụ trách:** Trần Ngọc Thanh Tú (Làm sạch & Chuẩn hóa văn bản/toán), Cáp Hửu Duy (BibTeX & Khử trùng lặp trích dẫn), Đặng Ngọc Hoàng (Kho phần tử dùng chung), Nguyễn Lê Anh Tuấn (Tổng hợp & Đánh giá định lượng)  
> **Mục tiêu:** Loại bỏ định dạng LaTeX không mang nghĩa, quy chuẩn hóa công thức toán, thực thi cơ chế băm nội dung để khử trùng lặp tự động liên phiên bản, chuyển đổi `\bibitem` sang BibTeX chuẩn, hợp nhất trích dẫn đa phiên bản bằng thuật toán gom cụm bắc cầu, và đo lường định lượng trên tập dữ liệu đối sánh tự nhiên từ arXiv.

---

### Vai trò cốt lõi của chuẩn hóa đối với khử trùng lặp dữ liệu

Đề bài nhấn mạnh: *"full-text matching should be performed after cleanup to remove discrepancies in minor details such as formatting or spacing that might otherwise cause the code to identify identical texts as different"*.

Nếu hai phiên bản v1 và v2 của một bài báo có chung một câu nhưng chỉ khác nhau một dấu cách thừa, một lệnh in đậm `\textbf{...}` hoặc khác tên citation key (`\cite{vaswani2017}` vs `\cite{vit_transformer}`), việc không chuẩn hóa sẽ khiến hàm băm tạo ra hai `element_id` hoàn toàn khác nhau. Khi đó, tỷ lệ khử trùng lặp sẽ sụp đổ. Do đó, **chuẩn hóa văn bản, đồng bộ công thức và chuẩn hóa khóa trích dẫn là điều kiện tiên quyết quyết định sự thành bại của giải thuật khử trùng lặp**.

In [1]:
import os
import sys
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

data_dir = Path("../data") if Path("../data").exists() else Path("data")

# Nạp dữ liệu khử trùng lặp trích dẫn từ Cáp Hửu Duy
df_bib_dedup = pd.read_csv(data_dir / "bib_dedup.csv") if (data_dir / "bib_dedup.csv").exists() else pd.read_csv(data_dir / "duy_m2_bib_dedup.csv")

with open(data_dir / "cite_rename_map.json", "r", encoding="utf-8") as f:
    cite_rename_map = json.load(f)

total_pubs_dedup = len(df_bib_dedup)
total_entries_before = df_bib_dedup['n_before'].sum()
total_entries_after = df_bib_dedup['n_after'].sum()
total_entries_reduced = total_entries_before - total_entries_after
dedup_ratio = total_entries_reduced / total_entries_before
total_cites_renamed = df_bib_dedup['n_cite_renamed'].sum()

print("===============================================================================")
print("TỔNG KẾT KHỬ TRÙNG LẶP TRÍCH DẪN & ĐỒNG BỘ KHÓA TRÍCH DẪN (HỬU DUY):")
print(f"- Số lượng bài báo thực hiện khử trùng lặp     : {total_pubs_dedup:,} bài")
print(f"- Tổng số mục BibTeX ban đầu (Before Dedup)     : {total_entries_before:,} entries")
print(f"- Số mục BibTeX duy nhất sau khi gộp (After)    : {total_entries_after:,} entries")
print(f"- Số lượng bản ghi trùng lặp được loại bỏ       : {total_entries_reduced:,} entries")
print(f"- TỶ LỆ KHỬ TRÙNG LẶP TRÍCH DẪN (DEDUP RATIO)   : {dedup_ratio*100:.2f}%")
print(f"- Tổng số lệnh \\cite được đồng bộ lại tên khóa : {total_cites_renamed:,} vị trí")
print(f"- Số bài báo có ánh xạ đổi tên citation key     : {len(cite_rename_map):,} bài")
print("===============================================================================")

TỔNG KẾT KHỬ TRÙNG LẶP TRÍCH DẪN & ĐỒNG BỘ KHÓA TRÍCH DẪN (HỬU DUY):
- Số lượng bài báo thực hiện khử trùng lặp     : 306 bài
- Tổng số mục BibTeX ban đầu (Before Dedup)     : 90,705 entries
- Số mục BibTeX duy nhất sau khi gộp (After)    : 45,652 entries
- Số lượng bản ghi trùng lặp được loại bỏ       : 45,053 entries
- TỶ LỆ KHỬ TRÙNG LẶP TRÍCH DẪN (DEDUP RATIO)   : 49.67%
- Tổng số lệnh \cite được đồng bộ lại tên khóa : 2,437 vị trí
- Số bài báo có ánh xạ đổi tên citation key     : 102 bài


### 1. Quy trình làm sạch văn bản và chuẩn hóa công thức toán học

Module `src/parser/normalize.py` do **Trần Ngọc Thanh Tú** phụ trách thực hiện hai công đoạn chuẩn hóa kế tiếp nhau:

#### A. Làm sạch văn bản (`clean_latex`)
- **Loại bỏ tham số định dạng:** Xóa các tùy chọn canh vị trí ảnh/bảng không mang nghĩa ngữ nghĩa (ví dụ: `[htbp!]`, `[width=0.8\textwidth]`).
- **Bóc tách giữ lõi văn bản:** Giữ lại nội dung bên trong các lệnh nhấn mạnh: `\textbf{nội dung}` $\rightarrow$ `nội dung`, `\textit{nội dung}` $\rightarrow$ `nội dung`, `\emph{nội dung}` $\rightarrow$ `nội dung`.
- **Xóa bỏ lệnh định dạng khung/trình bày:** Loại bỏ hoàn toàn `\centering`, `\noindent`, `\midrule`, `\toprule`, `\bottomrule`, `\hline`.
- **Quy chuẩn khoảng trắng:** Rút gọn toàn bộ chuỗi khoảng trắng thừa và ký tự xuống dòng tab về đúng một dấu cách đơn (`[ \t]+` $\rightarrow$ `" "`).
- **Tính bất biến (Idempotency):** Đảm bảo tính chất toán học $f(f(x)) = f(x)$, chạy làm sạch nhiều lần không làm thay đổi kết quả.

#### B. Chuẩn hóa công thức toán (`normalize_math`)
- **Toán khối (Block Math):** Quy đổi toàn bộ cú pháp TeX cổ `$$...$$`, TeX nguyên thủy `\[...\]`, cùng các môi trường `align*`, `gather*`, `eqnarray*` về dạng thống nhất duy nhất: `\begin{equation}...\end{equation}`.
- **Toán nội dòng (Inline Math):** Quy đổi toàn bộ cú pháp `\(...\)` về dạng chuẩn `$...$`.

In [2]:
# Thử nghiệm trực tiếp tính ổn định và chuẩn hóa công thức toán
import re

def demo_clean(s: str) -> str:
    s = re.sub(r'\\(?:textbf|textit|emph)\{([^}]*)\}', r'\1', s)
    s = re.sub(r'\\(?:centering|noindent|midrule|toprule|bottomrule|hline)', ' ', s)
    s = re.sub(r'\[[htbp!]+\]', '', s)
    s = re.sub(r'[ \t]+', ' ', s)
    return s.strip()

def demo_normalize_math(s: str) -> str:
    # Block math -> equation
    s = re.sub(r'\$\$(.*?)\$\$', r'\\begin{equation}\1\\end{equation}', s, flags=re.DOTALL)
    s = re.sub(r'\\\[(.*?)\\\]', r'\\begin{equation}\1\\end{equation}', s, flags=re.DOTALL)
    s = re.sub(r'\\begin\{(?:align\*?|eqnarray\*?|gather\*?)\}(.*?)\\end\{(?:align\*?|eqnarray\*?|gather\*?)\}',
               r'\\begin{equation}\1\\end{equation}', s, flags=re.DOTALL)
    # Inline math \(..\) -> $..$
    s = re.sub(r'\\\((.*?)\\\)', r'$\1$', s, flags=re.DOTALL)
    return s

raw_text = r"\centering In Eq. \[ E = mc^2 \], we observe \textbf{high accuracy} with \( O(N) \) complexity."
step1 = demo_clean(raw_text)
step2 = demo_normalize_math(step1)
step3 = demo_clean(demo_normalize_math(step2))

print("KIỂM ĐỊNH TÍNH CHUẨN HÓA VÀ TÍNH ỔN ĐỊNH (IDEMPOTENCY):")
print(f"- Văn bản thô ban đầu  : {raw_text}")
print(f"- Sau làm sạch & chuẩn : {step2}")
print(f"- Kiểm định lặp lại    : {step2 == step3} (Thỏa mãn tính chất f(f(x)) = f(x))")

KIỂM ĐỊNH TÍNH CHUẨN HÓA VÀ TÍNH ỔN ĐỊNH (IDEMPOTENCY):
- Văn bản thô ban đầu  : \centering In Eq. \[ E = mc^2 \], we observe \textbf{high accuracy} with \( O(N) \) complexity.
- Sau làm sạch & chuẩn : In Eq. \begin{equation} E = mc^2 \end{equation}, we observe high accuracy with $ O(N) $ complexity.
- Kiểm định lặp lại    : True (Thỏa mãn tính chất f(f(x)) = f(x))


### 2. Cơ chế băm định danh phần tử (`element_uid`) và khử trùng lặp nội dung

Theo đặc tả tại `docs/schema_m2.md`, mỗi phần tử nội dung được cấp một khóa định danh toàn cục (`element_id`) sinh tự động thông qua hàm băm SHA-1 trên nội dung đã qua chuẩn hóa:

$$\text{element\_id} = \text{pub\_key} \,\#\, \text{kind\_code} \,\_\, \text{sha1}(\text{clean\_content})[:8]$$

Cấu trúc định danh gồm ba thành phần:
- `pub_key`: Mã ấn phẩm duy nhất (ví dụ `2507-06659`).
- `kind_code`: Ký hiệu đại diện phân loại (`sec` cho phân mục, `snt` cho câu văn, `eqn` cho công thức, `fig` cho hình ảnh/bảng).
- `sha1(...)[:8]`: 8 ký tự thập lục phân đầu tiên của chuỗi băm SHA-1.

**Đặc tính khử trùng lặp tự động:**
Khi hai phiên bản (ví dụ v1 và v2) của cùng một bài báo có một câu văn hoặc công thức toán học giống hệt nhau sau bước làm sạch, hàm băm sẽ trả về **cùng một giá trị `element_id`**.
Bảng `elements` chỉ lưu đúng 1 dòng duy nhất cho phần tử này. Bảng `hierarchy` của v1 và v2 chỉ cần trỏ tới `element_id` dùng chung đó $\rightarrow$ **khử trùng lặp tự nhiên mà không cần bước so khớp toàn bộ cặp tốn kém**.

In [3]:
# Mô phỏng thuật toán sinh element_uid và khử trùng lặp nội dung
def element_uid(pub_key: str, kind: str, clean_content: str) -> str:
    KIND_CODE = {
        "document": "doc", "section": "sec", "subsection": "sub",
        "paragraph": "par", "sentence": "snt", "formula": "eqn", "figure": "fig"
    }
    payload = f"{kind}\x00{clean_content}".encode("utf-8")
    h = hashlib.sha1(payload).hexdigest()[:8]
    return f"{pub_key}#{KIND_CODE.get(kind, 'elt')}_{h}"

pub = "2507-06659"
s_v1 = "We propose a novel attention mechanism for sequential modeling."
s_v2 = "We propose a novel attention mechanism for sequential modeling."  # Giống hệt
s_v3 = "We propose an adaptive attention mechanism for sequential modeling." # Chỉnh sửa

id_v1 = element_uid(pub, "sentence", s_v1)
id_v2 = element_uid(pub, "sentence", s_v2)
id_v3 = element_uid(pub, "sentence", s_v3)

print("KẾT QUẢ KHỬ TRÙNG LẶP PHẦN TỬ LIÊN PHIÊN BẢN:")
print(f"- v1 câu ID : {id_v1}")
print(f"- v2 câu ID : {id_v2} -> Khớp tuyệt đối: {id_v1 == id_v2} (Dedup thành công: chỉ lưu 1 bản ghi elements)")
print(f"- v3 câu ID : {id_v3} -> Nội dung sửa đổi sinh ID mới: {id_v1 != id_v3}")

KẾT QUẢ KHỬ TRÙNG LẶP PHẦN TỬ LIÊN PHIÊN BẢN:
- v1 câu ID : 2507-06659#snt_7bb50eeb
- v2 câu ID : 2507-06659#snt_7bb50eeb -> Khớp tuyệt đối: True (Dedup thành công: chỉ lưu 1 bản ghi elements)
- v3 câu ID : 2507-06659#snt_af7b9970 -> Nội dung sửa đổi sinh ID mới: True


### 3. Bóc tách trích dẫn `\bibitem` bằng biểu thức chính quy

Đề bài quy định: *"For citations defined using `\bibitem` within the `.tex` files, students must convert them into standard BibTeX entries using programmatic tools such as regular expressions."*

Module `src/parser/bib.py` do **Cáp Hửu Duy** phụ trách triển khai:
- `split_bibitems(tex: str)`: Tách các khối trích dẫn độc lập dựa trên mẫu regex `\\bibitem\s*(?:\[[^\]]*\])?\s*\{([^}]+)\}`.
- `parse_bibitem(key: str, raw: str)`: Bóc tách các trường thuộc tính từ văn bản thô phi cấu trúc:
  + `year`: Tìm kiếm số có 4 chữ số trong khoảng năm hợp lệ 19xx–20xx.
  + `doi`: Regex nhận dạng chuẩn mã định danh số đối tượng `10.\d{4,9}/[-._;()/:A-Za-z0-9]+`.
  + `arxiv_id`: Bắt cả hai chuẩn mã: chuẩn mới từ 2007 (`2310.12345`) và chuẩn chuyên ngành cũ (`hep-th/9901001`).
  + `title`: Bóc tách thông qua các định dạng nhấn mạnh `\emph{...}`, dấu nháy kép `''...''` hoặc macro tạp chí `\bibinfo{title}{...}` kèm thuật toán cân bằng dấu ngoặc nhọn lồng nhau.
  + **Bảo tồn văn bản gốc:** Luôn lưu trữ trường `raw` để phục vụ đối chiếu và kiểm chứng.

In [4]:
# Biểu đồ tương tác Plotly: Hiệu quả khử trùng lặp trích dẫn BibTeX (Hửu Duy)
fig_dedup = go.Figure(data=[
    go.Bar(name='Trước khử trùng lặp (Before)', x=['Tổng số Entry BibTeX'], y=[total_entries_before],
           marker_color='#1f77b4', text=[f"{total_entries_before:,}"], textposition='auto'),
    go.Bar(name='Sau khử trùng lặp (After)', x=['Tổng số Entry BibTeX'], y=[total_entries_after],
           marker_color='#2ca02c', text=[f"{total_entries_after:,}"], textposition='auto')
])
fig_dedup.update_layout(
    barmode='group',
    title=f"Hiệu quả khử trùng lặp trích dẫn đa phiên bản (Giảm {dedup_ratio*100:.2f}% trên 306 bài báo)",
    yaxis_title="Số lượng bản ghi BibTeX",
    height=420,
    margin=dict(t=50, b=30, l=40, r=20)
)
fig_dedup.show()

### 4. Thuật toán hợp nhất trích dẫn liên phiên bản (Unionize Fields) & Đồng bộ khóa `\cite`

Đề bài yêu cầu: *"unionize the fields rather than selecting one entry while discarding all other information"*.

Khi cùng một tài liệu xuất hiện ở nhiều phiên bản với các khóa citation key khác nhau (ví dụ: v1 dùng `vaswani2017`, v2 đổi thành `vit_transformer`):
1. **Chữ ký định danh (`signature`):** Nhận diện cùng một tài liệu thông qua `doi` hoặc bộ chuẩn hóa `(title_norm, authors_norm, year)`.
2. **Gom cụm bắc cầu (Transitive Clustering):** Nối nhóm tài liệu theo quan hệ bắc cầu, giải quyết triệt để các trường hợp v1 có năm nhưng thiếu DOI, còn v2 có DOI nhưng thiếu năm.
3. **Quy tắc chọn khóa chuẩn (Canonical Key):** Ưu tiên theo thứ tự tất định: Version xuất hiện sớm nhất $\rightarrow$ Key ngắn nhất $\rightarrow$ Thứ tự chữ cái.
4. **Hợp nhất trường thông tin (Unionize):** Lấy hợp toàn bộ các trường không rỗng giữa các phiên bản. Bản ghi hợp nhất cuối cùng giữ được cả `year` từ v1 VÀ `doi` từ v2.
5. **Đổi tên khóa `\cite` trong văn bản (`rename_cites`):**
   Được thực thi **trước** khi băm `element_uid`. Đảm bảo các câu văn trích dẫn ở v1 và v2 được ánh xạ về cùng một mã khóa chuẩn, giúp câu văn đồng nhất hoàn toàn và dedup thành công.

In [5]:
# Minh họa trích xuất thực tế từ cite_rename_map.json
sample_arxiv_id = "2503.11982"
if sample_arxiv_id in cite_rename_map:
    info = cite_rename_map[sample_arxiv_id]
    print(f"VÍ DỤ ĐỒNG BỘ KHÓA TRÍCH DẪN TRÊN BÀI BÁO {sample_arxiv_id}:")
    print(f"- Số entry ban đầu : {info['n_before']} -> Sau gộp: {info['n_after']}")
    print(f"- Các cặp citation key được hợp nhất về chuẩn:")
    sample_renames = {k: v for k, v in info['rename_map'].items() if k != v}
    for old_k, new_k in list(sample_renames.items())[:6]:
        print(f"  + '{old_k}'  --->  '{new_k}'")
else:
    print(f"Bài báo {sample_arxiv_id} không có trong bản đồ đổi tên.")

VÍ DỤ ĐỒNG BỘ KHÓA TRÍCH DẪN TRÊN BÀI BÁO 2503.11982:
- Số entry ban đầu : 158 -> Sau gộp: 149
- Các cặp citation key được hợp nhất về chuẩn:
  + 'Editor00'  --->  'Harel79'
  + 'Editor00a'  --->  'Harel79'
  + 'Smith10'  --->  'Novak03'
  + 'VanGundy08'  --->  'VanGundy07'
  + 'VanGundy09'  --->  'VanGundy07'
  + 'Conti:2009:DDS:1555009.1555162'  --->  '1555162'


### 5. Đánh giá khách quan trên tập đối sánh tự nhiên từ arXiv (124 version)

Để đánh giá độ chính xác của bộ trích xuất regex một cách độc lập và khách quan, nhóm khai thác **tập dữ liệu đối sánh tự nhiên gồm 124 phiên bản** (dải ID `2503`) chứa đồng thời cả tệp `.bib` gốc của tác giả và khối `\bibitem` trong văn bản TeX:
- Tệp `.bib` chuẩn do tác giả nộp đóng vai trò là tập đối chứng chuẩn hóa (Ground Truth).
- Thu thập **2.788 cặp đối ứng** có cùng `citation_key` giữa hai nguồn và đo lường tỷ lệ trùng khớp trường dữ liệu.

#### Kết quả đo lường độ đồng thuận liên nguồn (Concordance Table):

| Trường thông tin | Số cặp khớp | Mẫu số (Cả 2 nguồn có) | Tỷ lệ khớp / Cặp có trường | Tỷ lệ / Toàn bộ 2.788 cặp | Nhận xét kỹ thuật |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **DOI (`doi`)** | **904** | 906 | **99,78%** | 32,42% | Định danh số chuẩn xác gần như tuyệt đối |
| **Mã arXiv (`arxiv_id`)** | **255** | 256 | **99,61%** | 9,15% | Bắt chính xác cả chuẩn mới lẫn mã chuyên ngành cũ |
| **Năm xuất bản (`year`)** | **2.635** | 2.692 | **97,88%** | 94,51% | Trích xuất năm đạt độ phủ trên 94% toàn tập |
| **Họ tác giả đầu** | **2.434** | 2.712 | **89,75%** | 87,30% | Nhận diện tác giả đạt độ tin cậy cao |
| **Tiêu đề chính xác (Exact)** | **1.511** | 1.639 | **92,19%** | 54,20% | Trùng khớp từng ký tự trên 1.639 cặp có tiêu đề |
| **Tiêu đề mềm (Soft Match)** | **1.524** | 1.639 | **92,98%** | 54,66% | Khớp chính xác hoặc bao hàm chuỗi con |

> **Ghi chú phản biện trung thực:** Có 1.149 cặp (41,2%) thiếu tiêu đề ở một trong hai nguồn. Nguyên nhân khách quan xuất phát từ quy ước xuất bản của các tạp chí Vật lý/Toán học (như APS, Springer) thường in `\bibitem` dạng rút gọn, hoàn toàn không in tên bài báo để tiết kiệm diện tích trang in. Module `collect_bib()` ưu tiên lấy bản ghi từ `.bib` khi phát hiện trùng key để khắc phục hoàn toàn hiện tượng này.

In [6]:
# Biểu đồ tương tác Plotly: Tỷ lệ khớp trường trích dẫn trên tập đối sánh tự nhiên (Hửu Duy)
fields = ['DOI (doi)', 'Mã arXiv', 'Năm (year)', 'Tiêu đề exact', 'Tiêu đề soft', 'Tác giả đầu']
match_rates = [99.78, 99.61, 97.88, 92.19, 92.98, 89.75]

fig_eval = px.bar(
    x=match_rates,
    y=fields,
    orientation='h',
    text=[f"{r:.2f}%" for r in match_rates],
    title='Tỷ lệ khớp trường dữ liệu giữa regex \\bibitem và tệp .bib chuẩn (2.788 cặp)',
    labels={'x': 'Tỷ lệ trùng khớp (%)', 'y': 'Trường dữ liệu'},
    color=match_rates,
    color_continuous_scale='Teal'
)
fig_eval.update_traces(textposition='inside', insidetextanchor='middle')
fig_eval.update_layout(xaxis=dict(range=[70, 103]), height=380, margin=dict(t=50, b=30, l=120, r=20))
fig_eval.show()